### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part5_rl_tests.py

import mugrade
import os


from part5_rl_tests import *
os.environ["MUGRADE_HW"] = "Part 5 - Reinforcement Learning"
os.environ["MUGRADE_KEY"] = "" ### Your key here

In [ ]:
### Download the necessary files
from huggingface_hub import hf_hub_download
import os

filenames = [
    'llm.d30.pt',
    'tokenizer_50M.bpe',
    'tokenizer_50M_rl.bpe',
    'config.d30.json',
    'config.d30.sft.json',
    'config.d30.rl.json',
    'config.d30.eval.json',
    'gsm8k_train.json',
    'gsm8k_test.json',
]
for f in filenames:
    if not os.path.exists(f):
        hf_hub_download(repo_id="zkolter/llm_speedrun", filename=f, repo_type="dataset", local_dir=".")

### BPE

Copy your BPE implementation from previous parts.  One suggestion: if you do want to run the full RL training runs, you'll want to edit the `decode()` function as shown in class to only decode tokens that are within the length of the vocabulary (with that much generation, there is a substantial probability the LLM generates a token outside this count, since we round the output matrix to the nearest larger multiple of 256).

In [ ]:
import json
import re

### BEGIN YOUR CODE

### END YOUR CODE


### LLM Architecture

Copy your LLM architecture from the previous assignment on inference.  Only the `generate()` function is tested for this assignment, which you should augment from the previous assignment to include tool calls.
The student notebook has a plain paste field for the previous model code. Implement the separate `generate(self, ...)` function below it; the provided assignment attaches it to your pasted model. Its `self` interface is the same as an instance method.


In [ ]:
import torch
import math
import random

### BEGIN YOUR CODE

### END YOUR CODE

class LLM:
    ### BEGIN YOUR CODE

    ### END YOUR CODE

    # @mugrade.local_tests
    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None, tool_eval=None, bpe=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE



### LLM Supervised finetuning

The following code augments the tokenizer with the needed tags and downloads and formats the GSM8K dataset.  You don't actually need to run this, as and we provide the fixed tokenizer and dataset files, but it is here for completeness.

In [ ]:
bpe = BPE("tokenizer_50M.bpe")
vocab_size = len(bpe.vocab)
bpe.special_tokens[vocab_size] = "<QUESTION>"
bpe.special_tokens[vocab_size+1] = "</QUESTION>"
bpe.special_tokens[vocab_size+2] = "<THINK>"
bpe.special_tokens[vocab_size+3] = "</THINK>"
bpe.special_tokens[vocab_size+4] = "<TOOL>"
bpe.special_tokens[vocab_size+5] = "</TOOL>"
bpe.special_tokens[vocab_size+6] = "<RESPONSE>"
bpe.special_tokens[vocab_size+7] = "</RESPONSE>"
bpe.special_tokens[vocab_size+8] = "<ANSWER>"
bpe.special_tokens[vocab_size+9] = "</ANSWER>"
bpe.special_tokens[vocab_size+10] = "<MASK/>"

bpe.vocab += [""]*(max(bpe.special_tokens.keys())-len(bpe.vocab)+1)
for k,v in bpe.special_tokens.items():
    bpe.vocab[k] = v
bpe.save("tokenizer_50M_rl.bpe")

from datasets import load_dataset
import random
max_seq_len = 512+1 # max actual GSM8K size is 400, so add additional padding
mask_token = bpe.vocab.index("<MASK/>")

for split in ["train", "test"]:
    tokens = []
    for example in load_dataset("openai/gsm8k", "main", split=split):
        question = example["question"]
        think, answer = example["answer"].split("\n#### ")
        for toolcall in re.findall(r"<<.+?>>", think):
            call, resp = toolcall[2:-2].split("=")
            think = think.replace(toolcall, f"<TOOL>{call}</TOOL><RESPONSE>{resp}</RESPONSE>")
        text = f"<DOCUMENT><QUESTION>{question}</QUESTION><THINK>{think}</THINK><ANSWER>{answer.replace(",","")}</ANSWER></DOCUMENT>"
        tokens.append({"tokens":bpe.encode(text), "text":text})
        tokens[-1]["tokens"] += [mask_token]*(max_seq_len-len(tokens[-1]["tokens"]))
    random.seed(42)
    random.shuffle(tokens)
    with open(f"gsm8k_{split}.json", "wt") as f:
        json.dump(tokens, f)

Implement the functions needed for supervised finetuning.  You will need to augment the cross entropy loss function from previous assignments to allow for a mask token (where loss is not applied) and a weight term, that can weight each element of the computed loss.  We'll take the convention you should still normalize by the total number of non-masked tokens (regardless of the weight term).  Implement Adam exactly like in the previous portions but _without_ a learning rate schedule.  Finally, implement supervised finetuning.  As in class, you should randomly pad inputs with some number of `<MASK/>` tokens, so that the model learns to ignore mask tokens at the start of a sequence (so that you can later do parallel generation over multiple prompts).


In [ ]:
import random

# @mugrade.local_tests
def cross_entropy_loss(logits, y, mask_token=-1, weights=1):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

class Adam:
    # @mugrade.local_tests
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def step(self):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

# @mugrade.local_tests
def finetune_llm_sft(rank, nccl_uid, config):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE



Once you have implemented the above, you should be able to finetune a model with the following code (not tested, only if you have a GPU and want to run it).

In [ ]:
from joblib import Parallel, delayed
import os
os.environ["NCCL_NVLS_ENABLE"] = "0"

with open("config.d30.sft.json", "rt") as f: config = json.load(f)
nccl_uid = torch.cuda.nccl.unique_id()
Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(finetune_llm_sft)(i, nccl_uid, config) for i in range(config["num_gpus"])
)

### Evaluating task performance

Once you have finetuned the model, you need to now use generation and tool calling to actually generate responses to problems and evaluate the performance.  Write the functions to call the tool (as we did in class, just by calling `eval()`), to extract answers from a response or ground truth, to grade correctness and correct formatting of responses, to evaluate a set of answers, and finally to evaluate a full dataset.

In [ ]:
# @mugrade.local_tests
def tool_eval(text):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

# @mugrade.local_tests
def extract_answer(text):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

# @mugrade.local_tests
def grade(responses, ground_truths):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

# @mugrade.local_tests
def eval_examples(llm, examples, bpe, tool_eval, k, seq_len):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


# @mugrade.local_tests
def eval_dataset(rank, config):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE



After this implementation, the following code will evaluate the sft-trained model.

In [ ]:
with open("config.d30.eval.json", "rt") as f: config = json.load(f)
config["filename"] = "llm.d30.sft.pt"
out = Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(eval_dataset)(i, config) for i in range(config["num_gpus"])
)
pass_at_1, pass_at_k, valid_at_1 = [sum(o[i] for o in out) for i in range(3)]
print(f"Pass@1: {pass_at_1}, Pass@{config["k"]}: {pass_at_k}, Valid: {valid_at_1}")


### Training with RL

Finally, adapt the SFT training function to include RL training.  As demonstrated in class, given all the functions above, this actually represents a fairly straightforward modification of the SFT finetuning code.


In [ ]:
# @mugrade.local_tests
def finetune_llm_rl(rank, nccl_uid, config):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

After this code, the following cells will RL train the model then evaluate it.

In [ ]:
from joblib import Parallel, delayed
import os
os.environ["NCCL_NVLS_ENABLE"] = "0"

with open("config.d30.rl.json", "rt") as f: config = json.load(f)
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(finetune_llm_rl)(i, nccl_uid, config) for i in range(config["num_gpus"])
)

In [ ]:
with open("config.d30.eval.json", "rt") as f: config = json.load(f)
out = Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(eval_dataset)(i, config) for i in range(config["num_gpus"])
)
pass_at_1, pass_at_k, valid_at_1 = [sum(o[i] for o in out) for i in range(3)]
print(f"Pass@1: {pass_at_1}, Pass@{config["k"]}: {pass_at_k}, Valid: {valid_at_1}")